In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('data_combined.csv', low_memory=False)

if 'AssociationFee' in df.columns:
    df['AssociationFee'] = df['AssociationFee'].fillna(0)
if 'GarageSpaces' in df.columns:
    df['GarageSpaces'] = df['GarageSpaces'].fillna(0)
cols_to_drop_na = ['LotSizeSquareFeet', 'YearBuilt', 'LivingArea', 
                   'Latitude', 'Longitude', 'BathroomsTotalInteger', 
                   'ClosePrice']
df = df.dropna(subset=[c for c in cols_to_drop_na if c in df.columns])

df['CloseDate'] = pd.to_datetime(df['CloseDate'], errors='coerce')
cutoff_date = pd.to_datetime('2023-06-01')
df = df[df['CloseDate'] >= cutoff_date].copy()

In [3]:
max_date = df['CloseDate'].max()
test_start_date = max_date - pd.DateOffset(months=1)
train_start_date = test_start_date - pd.DateOffset(months=12)

train_mask = (df['CloseDate'] > train_start_date) & (df['CloseDate'] <= test_start_date)
test_mask = (df['CloseDate'] > test_start_date) & (df['CloseDate'] <= max_date)

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()


In [4]:
coords_train = df_train[['Latitude', 'Longitude']]
coords_test = df_test[['Latitude', 'Longitude']]

kmeans = KMeans(n_clusters=50, random_state=42, n_init='auto')
df_train['Geo_Cluster'] = kmeans.fit_predict(coords_train)
df_test['Geo_Cluster'] = kmeans.predict(coords_test)


In [6]:
cluster_target_means = df_train.groupby('Geo_Cluster')['ClosePrice'].apply(lambda x: np.mean(np.log1p(x))).to_dict()

df_train['Cluster_Avg_LogPrice'] = df_train['Geo_Cluster'].map(cluster_target_means)
df_test['Cluster_Avg_LogPrice'] = df_test['Geo_Cluster'].map(cluster_target_means)

global_mean = np.mean(np.log1p(df_train['ClosePrice']))
df_test['Cluster_Avg_LogPrice'] = df_test['Cluster_Avg_LogPrice'].fillna(global_mean)


In [7]:
features = ['LivingArea', 'LotSizeSquareFeet', 'BedroomsTotal', 
            'BathroomsTotalInteger', 'YearBuilt', 'AssociationFee', 
            'GarageSpaces', 'Cluster_Avg_LogPrice']

In [9]:
df_train_final = df_train[features + ['ClosePrice']].dropna()
df_test_final = df_test[features + ['ClosePrice']].dropna()

In [10]:
import pandas as pd

df_train_final['Set_Type'] = 'Train'
df_test_final['Set_Type'] = 'Test'

df_golden = pd.concat([df_train_final, df_test_final], axis=0)


cols = [c for c in df_golden.columns if c != 'ClosePrice'] + ['ClosePrice']
df_golden = df_golden[cols]


output_filename = 'California_Housing_TargetEncoded.csv'
df_golden.to_csv(output_filename, index=False)

print(f"Filename: {output_filename}")
print(f"Final Feature Count: {df_golden.shape[1] - 2}") 

Filename: California_Housing_TargetEncoded.csv
Final Feature Count: 8


now we have this light-weighted geocoding dataset, we could run the baseline model again

In [11]:
y_train = np.log1p(df_train_final['ClosePrice']) 
y_test = np.log1p(df_test_final['ClosePrice'])

X_train = df_train_final[features].copy()
X_test = df_test_final[features].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

train_pred = model.predict(X_train_scaled)
test_pred = model.predict(X_test_scaled)

print(f"Train R²: {r2_score(y_train, train_pred):.4f}")
print(f"Test R²:  {r2_score(y_test, test_pred):.4f}")
print(f" One-Hot Encoding Baseline Test R²: 0.7254")


Train R²: 0.7260
Test R²:  0.7272
 One-Hot Encoding Baseline Test R²: 0.7254


The current score is extremely close to 0.7254, indicating that we have successfully compressed 50 columns of sparse geographic features into a single column of highly condensed, high-quality features. The dimensionality of the data has been significantly reduced, and sparsity has been completely eliminated. The model achieved the same performance as the original 57 features using only 8 features, and the effectiveness of the subsequent tree model will also improve significantly.

Granth divided California into 10 major regions based on closing prices, while I used pure WCSS to divide California into 50 commercial districts. Both methods have their own advantages and disadvantages. His method lacks sufficient detail, and directly including the values 0–9 in the model can mislead it; whereas my approach, using 50 dummy variables, resulted in an extremely sparse matrix. Therefore, I combined the two approaches: I kept k=50 but converted each commercial district into its historical average log house price. This reduced the number of features from 57 to 8, allowing the tree model to perform at its best and most efficiently.